<a href="https://colab.research.google.com/github/douglasbaquiao/materials-solar-ml/blob/main/notebooks/v2_eda_campos_expandidos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Materials Informatics — EDA v2
## Perovskitas Duplas Halogenadas × Kesteritas para Aplicações Fotovoltaicas

**Versão:** 2.1 — reorganização e classificação por grupo espacial
**Diferenças em relação à v1:**
- Extração e feature engineering delegadas ao módulo `src/extraction.py`
- Campos expandidos: `efermi` (alta cobertura) + campos de cobertura parcial mantidos para fases futuras
- Classificação por grupo espacial esperado: `estrutura_esperada`
- Nomenclatura padronizada: "Perovskitas Duplas" (em lugar de "Double Perovskitas")
- Dados salvos em `data/processed/` e figuras em `figures/`
- Compatível com execução local e Google Colab

## 1. Configuração do Ambiente

In [ ]:
import subprocess, sys, os

# ── Instalação de dependências ────────────────────────────────────────────────
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "mp-api", "pymatgen", "python-dotenv", "-q"
])

# ── Detecção de ambiente ──────────────────────────────────────────────────────
try:
    from google.colab import userdata
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

print(f"Ambiente: {'Google Colab' if EM_COLAB else 'Local (Jupyter / VS Code)'}")

### 1.1 Chave de API

- **Colab:** configurar no painel *Secrets* (🔑 na barra lateral) com o nome `MP_API_KEY`.
- **Local:** arquivo `.env` na raiz do projeto ou variável de ambiente `MP_API_KEY`.
  O `.env` está no `.gitignore` e nunca é versionado.

In [ ]:
if EM_COLAB:
    os.environ["MP_API_KEY"] = userdata.get("MP_API_KEY")
    print("Chave carregada via Colab Secrets.")
else:
    print("Chave será lida do .env ou da variável de ambiente MP_API_KEY.")

## 2. Estrutura de Pastas do Repositório

```
materials-solar-ml/
├── src/extraction.py          ← módulo importado por este notebook
├── notebooks/
│   └── v2_eda_campos_expandidos.ipynb
├── data/
│   └── processed/             ← CSVs gerados aqui
└── figures/                   ← PNGs gerados aqui
```

In [ ]:
REPO_URL  = "https://github.com/douglasbaquiao/materials-solar-ml.git"
REPO_NAME = "materials-solar-ml"

if EM_COLAB:
    repo_path = f"/content/{REPO_NAME}"
    if not os.path.exists(repo_path):
        subprocess.run(["git", "clone", REPO_URL, repo_path])
        print(f"Repositório clonado em {repo_path}")
    else:
        subprocess.run(["git", "-C", repo_path, "pull"])
        print(f"Repositório atualizado: {repo_path}")
    os.chdir(repo_path)

RAIZ = os.getcwd()
print(f"Raiz do projeto: {RAIZ}")

PASTA_PROCESSED = os.path.join(RAIZ, "data", "processed")
PASTA_FIGURES   = os.path.join(RAIZ, "figures")

for pasta in [os.path.join(RAIZ,"data","raw"), PASTA_PROCESSED, PASTA_FIGURES]:
    os.makedirs(pasta, exist_ok=True)

print("Pastas verificadas:")
for p in [PASTA_PROCESSED, PASTA_FIGURES]:
    print(f"  {p}")

## 3. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.join(RAIZ, "src"))

from extraction import (
    conectar_api, extrair_familia, adicionar_features,
    pipeline_completo, exportar, carregar,
    PV_GAP_MIN, PV_GAP_MAX, IBSC_GAP_MAX, HULL_THRESH, FAMILIAS,
)

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# Nomenclatura e cores padronizadas para todo o notebook
FAMILIAS_PLOT = {
    "perovskita": {"label": "Perovskitas Duplas", "cor": "#E07B54"},
    "kesterita":  {"label": "Kesteritas",         "cor": "#4A90D9"},
}

print("Imports concluídos.")
print(f"Janela PV:   {PV_GAP_MIN}–{PV_GAP_MAX} eV")
print(f"Janela IBSC: {PV_GAP_MAX}–{IBSC_GAP_MAX} eV")
print(f"Hull thresh: {HULL_THRESH} eV/átomo")

## 4. Extração de Dados

**Primeira execução:** descomentar `pipeline_completo()`.  
**Execuções seguintes:** manter apenas os `carregar()`.

Os dados já incluem `efermi` (alta cobertura) e os demais campos v2
com cobertura parcial — disponíveis para análise dos candidatos finais.

In [ ]:
# ── OPÇÃO A: primeira execução ────────────────────────────────────────────────
# dados = pipeline_completo(exportar_=True, verbose=True)
# df_p  = dados["perovskita"]
# df_k  = dados["kesterita"]

# ── OPÇÃO B: execuções seguintes ──────────────────────────────────────────────
df_p = carregar("perovskita", pasta=PASTA_PROCESSED)
df_k = carregar("kesterita",  pasta=PASTA_PROCESSED)

print(f"Perovskitas Duplas: {len(df_p):>5} materiais, {len(df_p.columns)} colunas")
print(f"Kesteritas:         {len(df_k):>5} materiais, {len(df_k.columns)} colunas")

## 5. Cobertura dos Campos v2

Verifica quais campos têm dados suficientes para análise.
Campos com cobertura abaixo de ~20% são mantidos no dataset mas
reservados para análise individual dos candidatos finais (Fase 4).

In [ ]:
def cobertura_campos(df, nome):
    campos = [
        "efermi", "e_electronic", "e_ionic", "e_total", "n_refractive",
        "weighted_work_function", "weighted_surface_energy",
        "bulk_modulus_vrh", "shear_modulus_vrh",
        "has_dielectric_data", "has_elastic_data", "has_experimental_ref",
    ]
    print(f"\n── {nome} (n={len(df)}) ──")
    for campo in campos:
        if campo not in df.columns:
            continue
        if df[campo].dtype == bool:
            n = df[campo].sum()
        else:
            n = df[campo].notna().sum()
        pct   = 100 * n / len(df)
        barra = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
        print(f"  {campo:<35} {barra}  {n:>5}/{len(df)} ({pct:5.1f}%)")

cobertura_campos(df_p, "Perovskitas Duplas")
cobertura_campos(df_k, "Kesteritas")

## 6. Classificação por Grupo Espacial

A extração via API usa filtros composicionais (presença de haleto ou
calcogeneto + 4 elementos) como proxy das famílias pretendidas.
Isso captura compostos com física diferente — rutilos, fluoritas,
espinélios — que precisam ser identificados e separados antes da análise.

### Grupos espaciais de referência

**Perovskitas Duplas A₂B'B''X₆:**

| Nº | Símbolo | Sistema | Tipo |
|---|---|---|---|
| 225 | Fm-3m | Cúbico | Protótipo ideal (ex: Cs₂AgBiBr₆) |
| 139 | I4/mmm | Tetragonal | Distorção tetragonal comum |
| 123 | P4/mmm | Tetragonal | Distorção leve |
| 62 | Pnma | Ortorrômbico | Distorção Glazer a⁻b⁺a⁻ |
| 74 | Imma | Ortorrômbico | Distorção intermediária |
| 12 | C2/m | Monoclínico | Distorção severa — relacionada |
| 2 | P-1 | Triclínico | Alta distorção — relacionada |

**Kesteritas A₂BCX₄:**

| Nº | Símbolo | Sistema | Tipo |
|---|---|---|---|
| 82 | I-4 | Tetragonal | Kesterita (ordering Cu/Zn alternado) |
| 121 | I-42m | Tetragonal | Estannita |
| 119 | I-4m2 | Tetragonal | Kesterita primitiva |
| 122 | I-42d | Tetragonal | Calcopirita — relacionada |
| 160 | R3m | Trigonal | Variante desordenada — relacionada |

A coluna `estrutura_esperada` assume três valores:
- `"perovskita dupla"` / `"kesterita"` — grupo na lista canônica
- `"relacionada"` — grupo de família estruturalmente próxima
- `"outra"` — fora das listas

In [ ]:
# ── Definição dos grupos por família ─────────────────────────────────────────

GS_PEROVSKITA_DUPLA = {225, 139, 123, 62, 74}      # confirmadas
GS_PEROVSKITA_REL   = {12, 2}                       # relacionadas (alta distorção)

GS_KESTERITA        = {82, 121, 119}                # confirmadas
GS_KESTERITA_REL    = {122, 160}                    # relacionadas


def classificar_estrutura(spacegroup_number, gs_confirmados, gs_relacionados, nome_familia):
    # Retorna a categoria estrutural com base no número do grupo espacial.
    # nome_familia: string usada no rótulo confirmado (ex: 'perovskita dupla')
    if pd.isna(spacegroup_number):
        return "outra"
    sg = int(spacegroup_number)
    if sg in gs_confirmados:
        return nome_familia
    if sg in gs_relacionados:
        return "relacionada"
    return "outra"


# ── Aplicar classificação ─────────────────────────────────────────────────────

df_p["estrutura_esperada"] = df_p["spacegroup_number"].apply(
    lambda sg: classificar_estrutura(sg, GS_PEROVSKITA_DUPLA, GS_PEROVSKITA_REL, "perovskita dupla")
)

df_k["estrutura_esperada"] = df_k["spacegroup_number"].apply(
    lambda sg: classificar_estrutura(sg, GS_KESTERITA, GS_KESTERITA_REL, "kesterita")
)

# ── Resumo da classificação ───────────────────────────────────────────────────

print("Perovskitas Duplas — distribuição de estrutura_esperada:")
ct_p = df_p["estrutura_esperada"].value_counts()
for cat, n in ct_p.items():
    print(f"  {cat:<20} {n:>5}  ({100*n/len(df_p):.1f}%)")

print(f"\nKesteritas — distribuição de estrutura_esperada:")
ct_k = df_k["estrutura_esperada"].value_counts()
for cat, n in ct_k.items():
    print(f"  {cat:<20} {n:>5}  ({100*n/len(df_k):.1f}%)")

## 7. Análise por Categoria Estrutural

Compara as distribuições de propriedades entre as três categorias.
O objetivo é verificar se os materiais fora do grupo esperado
têm comportamento sistematicamente diferente — o que justificaria
usar apenas a categoria "confirmada" nas análises seguintes.

In [ ]:
def sumario_por_categoria(df, nome_familia):
    # Estatísticas descritivas segmentadas por estrutura_esperada
    metricas = ["band_gap", "formation_energy_per_atom",
                "energy_above_hull", "density", "efermi"]
    print(f"\n{'='*68}")
    print(f"{nome_familia.upper()} — Médias por categoria estrutural")
    print(f"{'='*68}")
    resumo = (
        df.groupby("estrutura_esperada")[metricas]
        .agg(["mean", "median", "count"])
        .round(3)
    )
    print(resumo.to_string())

    # Proporção de candidatos PV por categoria
    print(f"\n── Candidatos PV por categoria ──")
    for cat in df["estrutura_esperada"].unique():
        sub  = df[df["estrutura_esperada"] == cat]
        n_pv = sub["is_pv_candidate"].sum()
        pct  = 100 * n_pv / len(sub) if len(sub) > 0 else 0
        print(f"  {cat:<20} PV: {n_pv:>4}/{len(sub):<5} ({pct:.1f}%)")

sumario_por_categoria(df_p, "Perovskitas Duplas")
sumario_por_categoria(df_k, "Kesteritas")

In [ ]:
# ── Visualização: band gap por categoria estrutural ───────────────────────────

ORDEM_CATS = ["perovskita dupla", "kesterita", "relacionada", "outra"]
CORES_CATS  = {
    "perovskita dupla": "#E07B54",
    "kesterita":        "#4A90D9",
    "relacionada":      "#8B6CAD",
    "outra":            "#AAAAAA",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Distribuição de Band Gap por Categoria Estrutural",
             fontsize=13, fontweight="bold")

for ax, (chave, df) in zip(axes, [("perovskita", df_p), ("kesterita", df_k)]):
    label_familia = FAMILIAS_PLOT[chave]["label"]
    cats_presentes = [c for c in ORDEM_CATS if c in df["estrutura_esperada"].unique()]

    for cat in cats_presentes:
        sub   = df[(df["estrutura_esperada"] == cat) & (~df["is_metal"])]
        dados = sub["band_gap"].dropna()
        if len(dados) < 5:
            continue
        cor = CORES_CATS.get(cat, "#999999")
        ax.hist(dados, bins=40, alpha=0.5, color=cor,
                density=True, label=f"{cat} (n={len(dados)})")
        if len(dados) > 20:
            dados.plot.kde(ax=ax, color=cor, linewidth=2)

    ax.axvspan(PV_GAP_MIN, PV_GAP_MAX,   alpha=0.10, color="green")
    ax.axvspan(PV_GAP_MAX, IBSC_GAP_MAX, alpha=0.06, color="blue")
    ax.set_xlabel("Band Gap (eV)", fontsize=11)
    ax.set_ylabel("Densidade", fontsize=11)
    ax.set_title(label_familia, fontsize=12)
    ax.legend(fontsize=8)
    ax.set_xlim(0, 5)

plt.tight_layout()
caminho = os.path.join(PASTA_FIGURES, "fig_gap_por_estrutura.png")
plt.savefig(caminho, dpi=150, bbox_inches="tight")
print(f"Salvo: {caminho}")
plt.show()

In [ ]:
# ── Exportar datasets com a nova coluna ──────────────────────────────────────
# Atualiza os CSVs para que análises futuras já tenham a classificação

exportar(df_p, "perovskita", pasta=PASTA_PROCESSED)
exportar(df_k, "kesterita",  pasta=PASTA_PROCESSED)
print("CSVs atualizados com coluna estrutura_esperada.")

## 8. Análise Estatística — Dataset Completo

Estatísticas sobre o dataset completo (todas as categorias estruturais).
Use os resultados da Seção 7 para interpretar quanto do comportamento
aqui observado é da família pretendida vs. compostos misturados.

In [ ]:
METRICAS_CORE = [
    "band_gap", "formation_energy_per_atom", "energy_above_hull",
    "density", "nsites", "volume", "site_density", "efermi",
]

for titulo, metricas in [("Features principais + efermi", METRICAS_CORE)]:
    print(f"\n{'='*68}")
    print(f"{titulo.upper()}")
    for chave, df in [("perovskita", df_p), ("kesterita", df_k)]:
        nome = FAMILIAS_PLOT[chave]["label"]
        cols = [m for m in metricas if m in df.columns]
        print(f"\n── {nome} ──")
        print(df[cols].describe().round(3).to_string())

# ── Sumário de candidatos ─────────────────────────────────────────────────────
print(f"\n{'='*68}")
print("SUMÁRIO DE CANDIDATOS")
print(f"{'='*68}")

linhas = [
    ("Candidatos PV (1.0–1.8 eV)",   "is_pv_candidate"),
    ("Candidatos IBSC (1.8–2.6 eV)", "is_ibsc_candidate"),
    ("Termod. estáveis (hull=0)",     "is_stable"),
    ("Quasi-estáveis (<50 meV)",      "near_hull"),
    ("Com referência experimental",   "has_experimental_ref"),
    ("Teóricos (sem síntese)",        "theoretical"),
]

for descricao, col in linhas:
    print(f"\n  {descricao}")
    for chave, df in [("perovskita", df_p), ("kesterita", df_k)]:
        nome = FAMILIAS_PLOT[chave]["label"]
        if col not in df.columns:
            continue
        n   = df[col].sum()
        pct = 100 * n / len(df)
        print(f"    {nome:22s}: {n:>4}/{len(df):<5} ({pct:5.1f}%)")

## 9. Visualizações

### Fig. 1 — Distribuição de Band Gap (dataset completo)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for chave, df in [("perovskita", df_p), ("kesterita", df_k)]:
    cfg   = FAMILIAS_PLOT[chave]
    dados = df.loc[~df["is_metal"], "band_gap"].dropna()
    ax.hist(dados, bins=50, alpha=0.45, color=cfg["cor"],
            density=True, label=f"{cfg['label']} (n={len(dados)})")
    dados.plot.kde(ax=ax, color=cfg["cor"], linewidth=2.5)

ax.axvspan(PV_GAP_MIN,   PV_GAP_MAX,   alpha=0.12, color="green",
           label=f"Janela PV ({PV_GAP_MIN}–{PV_GAP_MAX} eV)")
ax.axvspan(PV_GAP_MAX,   IBSC_GAP_MAX, alpha=0.07, color="blue",
           label=f"Janela IBSC ({PV_GAP_MAX}–{IBSC_GAP_MAX} eV)")
ax.axvline(1.34, color="gold", linestyle="--", linewidth=1.5, label="GaAs (1.34 eV)")
ax.axvline(1.12, color="gray", linestyle=":",  linewidth=1.2, label="Si (1.12 eV)")

ax.set_xlabel("Band Gap (eV)", fontsize=12)
ax.set_ylabel("Densidade de probabilidade", fontsize=12)
ax.set_title("Distribuição de Band Gap — Não-metais", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.set_xlim(0, 5)

plt.tight_layout()
caminho = os.path.join(PASTA_FIGURES, "fig1_distribuicao_gap.png")
plt.savefig(caminho, dpi=150, bbox_inches="tight")
print(f"Salvo: {caminho}")
plt.show()

### Fig. 2 — Estabilidade Termodinâmica

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Estabilidade Termodinâmica (Energy Above Hull)",
             fontsize=13, fontweight="bold")
bins = np.linspace(0, 0.6, 50)

for ax, (chave, df) in zip(axes, [("perovskita", df_p), ("kesterita", df_k)]):
    cfg = FAMILIAS_PLOT[chave]
    ax.hist(df["energy_above_hull"].clip(upper=0.6), bins=bins,
            color=cfg["cor"], alpha=0.75, edgecolor="white", linewidth=0.3)
    ax.axvline(HULL_THRESH, color="red",   linestyle="--", linewidth=1.5,
               label=f"Quasi-estável ({HULL_THRESH} eV/át.)")
    ax.axvline(0.0,         color="green", linestyle="-",  linewidth=1.2,
               label="Hull = 0 (estável)")
    n_est   = (df["energy_above_hull"] == 0).sum()
    n_quasi = (df["energy_above_hull"] < HULL_THRESH).sum()
    ax.set_title(f"{cfg['label']}\nEstáveis: {n_est}  |  Quasi: {n_quasi}",
                 fontsize=11)
    ax.set_xlabel("Energy Above Hull (eV/átomo)", fontsize=11)
    ax.set_ylabel("Contagem", fontsize=11)
    ax.legend(fontsize=9)

plt.tight_layout()
caminho = os.path.join(PASTA_FIGURES, "fig2_estabilidade.png")
plt.savefig(caminho, dpi=150, bbox_inches="tight")
print(f"Salvo: {caminho}")
plt.show()

### Fig. 3 — Energia de Formação × Band Gap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Energia de Formação vs. Band Gap", fontsize=13, fontweight="bold")

for ax, (chave, df) in zip(axes, [("perovskita", df_p), ("kesterita", df_k)]):
    cfg = FAMILIAS_PLOT[chave]
    sub = df[~df["is_metal"]].dropna(
        subset=["band_gap", "formation_energy_per_atom", "energy_above_hull"])
    sc = ax.scatter(sub["band_gap"], sub["formation_energy_per_atom"],
                    c=sub["energy_above_hull"].clip(upper=0.5),
                    cmap="YlOrRd_r", s=20, alpha=0.7,
                    vmin=0, vmax=0.4, edgecolors="none")
    plt.colorbar(sc, ax=ax, label="Energy Above Hull (eV/átomo)")
    ax.axvspan(PV_GAP_MIN, PV_GAP_MAX,   alpha=0.10, color="green")
    ax.axvspan(PV_GAP_MAX, IBSC_GAP_MAX, alpha=0.06, color="blue")
    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.6)
    ax.set_xlabel("Band Gap (eV)", fontsize=11)
    ax.set_ylabel("Energia de Formação (eV/átomo)", fontsize=11)
    ax.set_title(f"{cfg['label']}  (n={len(sub)} não-metais)", fontsize=11)
    ymin = ax.get_ylim()[0]
    ax.text(1.4, ymin * 0.92, "PV",   fontsize=8, color="green", alpha=0.8)
    ax.text(2.1, ymin * 0.92, "IBSC", fontsize=8, color="blue",  alpha=0.8)

plt.tight_layout()
caminho = os.path.join(PASTA_FIGURES, "fig3_formacao_vs_gap.png")
plt.savefig(caminho, dpi=150, bbox_inches="tight")
print(f"Salvo: {caminho}")
plt.show()

### Fig. 4 — Sistema Cristalino

In [ ]:
sistemas_ordem = ["cubic","tetragonal","orthorhombic",
                  "hexagonal","trigonal","monoclinic","triclinic"]

fig, ax = plt.subplots(figsize=(12, 5))
x, width = np.arange(len(sistemas_ordem)), 0.38

for offset, (chave, df) in zip([-width/2, width/2],
                                [("perovskita", df_p), ("kesterita", df_k)]):
    cfg  = FAMILIAS_PLOT[chave]
    ct   = df["crystal_system"].str.lower().value_counts()
    vals = [ct.get(s, 0) for s in sistemas_ordem]
    bars = ax.bar(x + offset, vals, width, label=cfg["label"],
                  color=cfg["cor"], alpha=0.85, edgecolor="white")
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    str(v), ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([s.capitalize() for s in sistemas_ordem], fontsize=11)
ax.set_ylabel("Contagem", fontsize=11)
ax.set_title("Sistema Cristalino por Família", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)

plt.tight_layout()
caminho = os.path.join(PASTA_FIGURES, "fig4_sistemas_cristalinos.png")
plt.savefig(caminho, dpi=150, bbox_inches="tight")
print(f"Salvo: {caminho}")
plt.show()

### Fig. 5 — Nível de Fermi (efermi)

O `efermi` é o campo v2 com maior cobertura (~100%).
Indica se o material é naturalmente tipo-n (Fermi próximo à banda de condução)
ou tipo-p (Fermi próximo à banda de valência) — informação importante
para o design da heterojunção na célula solar.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Nível de Fermi por Categoria Estrutural",
             fontsize=13, fontweight="bold")

for ax, (chave, df) in zip(axes, [("perovskita", df_p), ("kesterita", df_k)]):
    cfg = FAMILIAS_PLOT[chave]
    cats_plot = [c for c in ORDEM_CATS if c in df["estrutura_esperada"].unique()]
    for cat in cats_plot:
        sub   = df[df["estrutura_esperada"] == cat]
        dados = sub["efermi"].dropna()
        if len(dados) < 5:
            continue
        cor = CORES_CATS.get(cat, "#999999")
        ax.hist(dados, bins=35, alpha=0.5, color=cor,
                density=True, label=f"{cat} (n={len(dados)})")
        if len(dados) > 20:
            dados.plot.kde(ax=ax, color=cor, linewidth=2)

    ax.set_xlabel("Nível de Fermi (eV)", fontsize=11)
    ax.set_ylabel("Densidade", fontsize=11)
    ax.set_title(cfg["label"], fontsize=12)
    ax.legend(fontsize=8)

plt.tight_layout()
caminho = os.path.join(PASTA_FIGURES, "fig5_efermi.png")
plt.savefig(caminho, dpi=150, bbox_inches="tight")
print(f"Salvo: {caminho}")
plt.show()

### Fig. 6 — Mapa de Correlação entre Features

In [ ]:
# Apenas features com cobertura suficiente para não esvaziar o dropna()
FEAT_CORR = [
    "band_gap", "formation_energy_per_atom", "energy_above_hull",
    "density", "nsites", "volume", "site_density",
    "efermi", "total_magnetization",
]
NOMES_CURTOS = [
    "band_gap", "E_form", "E_hull",
    "density", "nsites", "volume", "site_dens",
    "efermi", "magnetiz.",
]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Correlação de Pearson entre Features",
             fontsize=13, fontweight="bold")

for ax, (chave, df) in zip(axes, [("perovskita", df_p), ("kesterita", df_k)]):
    cfg  = FAMILIAS_PLOT[chave]
    cols = [f for f in FEAT_CORR if f in df.columns]
    nomes = [NOMES_CURTOS[FEAT_CORR.index(f)] for f in cols]
    corr  = df[cols].dropna().corr()
    corr.columns = nomes
    corr.index   = nomes
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, ax=ax, annot=True, fmt=".2f",
                annot_kws={"size": 9}, cmap="RdBu_r",
                center=0, vmin=-1, vmax=1, linewidths=0.5,
                cbar_kws={"shrink": 0.85, "label": "Pearson r"})
    ax.set_title(cfg["label"], fontsize=11)
    ax.tick_params(axis="x", rotation=40, labelsize=9)
    ax.tick_params(axis="y", rotation=0,  labelsize=9)

plt.tight_layout()
caminho = os.path.join(PASTA_FIGURES, "fig6_correlacoes.png")
plt.savefig(caminho, dpi=150, bbox_inches="tight")
print(f"Salvo: {caminho}")
plt.show()

### Fig. 7 — Candidatos PV e IBSC

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("Candidatos Promissores — PV e IBSC",
             fontsize=13, fontweight="bold")

pares = [
    (axes[0,0], "Perovskitas Duplas — Candidatos PV",   df_p, "pv"),
    (axes[0,1], "Kesteritas — Candidatos PV",           df_k, "pv"),
    (axes[1,0], "Perovskitas Duplas — Candidatos IBSC", df_p, "ibsc"),
    (axes[1,1], "Kesteritas — Candidatos IBSC",         df_k, "ibsc"),
]

for ax, titulo, df, tipo in pares:
    col_flag = "is_pv_candidate" if tipo == "pv" else "is_ibsc_candidate"
    xmin = PV_GAP_MIN - 0.1 if tipo == "pv" else PV_GAP_MAX - 0.1
    xmax = PV_GAP_MAX + 0.1 if tipo == "pv" else IBSC_GAP_MAX + 0.1

    sub = df[df[col_flag] & (df["formation_energy_per_atom"] < 0)].dropna(
        subset=["band_gap", "formation_energy_per_atom", "energy_above_hull"])

    if len(sub) == 0:
        ax.text(0.5, 0.5, "Sem candidatos", ha="center", va="center",
                transform=ax.transAxes, fontsize=12)
        ax.set_title(titulo, fontsize=11)
        continue

    sc = ax.scatter(sub["band_gap"], sub["formation_energy_per_atom"],
                    c=sub["energy_above_hull"], cmap="YlOrRd_r",
                    s=40, alpha=0.85, vmin=0, vmax=0.3,
                    edgecolors="gray", linewidths=0.3)
    plt.colorbar(sc, ax=ax, label="E above hull (eV/át.)")

    for _, row in sub.nsmallest(6, "energy_above_hull").iterrows():
        ax.annotate(row["formula"],
                    (row["band_gap"], row["formation_energy_per_atom"]),
                    fontsize=7, xytext=(6, 4), textcoords="offset points",
                    arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))

    ax.set_xlim(xmin, xmax)
    ax.axhline(0, color="gray", linewidth=0.7, linestyle="--")
    ax.set_xlabel("Band Gap (eV)", fontsize=10)
    ax.set_ylabel("E_form (eV/átomo)", fontsize=10)
    ax.set_title(f"{titulo}  (n={len(sub)})", fontsize=11)

plt.tight_layout()
caminho = os.path.join(PASTA_FIGURES, "fig7_candidatos_scatter.png")
plt.savefig(caminho, dpi=150, bbox_inches="tight")
print(f"Salvo: {caminho}")
plt.show()

## 10. Exportação dos Candidatos

Salva os candidatos em `data/processed/` incluindo a coluna `estrutura_esperada`.
Os candidatos exportados aqui serão o ponto de partida da **Fase 2**.

In [ ]:
COLS_CANDIDATOS = [
    "material_id", "formula", "band_gap", "is_gap_direct",
    "energy_above_hull", "formation_energy_per_atom",
    "crystal_system", "spacegroup_symbol", "spacegroup_number",
    "estrutura_esperada", "density", "efermi",
    "theoretical", "has_experimental_ref", "possible_species",
]

def extrair_candidatos(df, tipo):
    col  = "is_pv_candidate" if tipo == "pv" else "is_ibsc_candidate"
    cols = [c for c in COLS_CANDIDATOS if c in df.columns]
    return (
        df[df[col] & (df["formation_energy_per_atom"] < 0) &
           (df["energy_above_hull"] < HULL_THRESH)]
        .sort_values(["estrutura_esperada", "energy_above_hull"])
        [cols]
        .reset_index(drop=True)
    )

cand_pv = pd.concat([
    extrair_candidatos(df_p, "pv").assign(familia="Perovskita Dupla"),
    extrair_candidatos(df_k, "pv").assign(familia="Kesterita"),
], ignore_index=True)

cand_ibsc = pd.concat([
    extrair_candidatos(df_p, "ibsc").assign(familia="Perovskita Dupla"),
    extrair_candidatos(df_k, "ibsc").assign(familia="Kesterita"),
], ignore_index=True)

exportar(cand_pv,   "candidatos_pv",   pasta=PASTA_PROCESSED)
exportar(cand_ibsc, "candidatos_ibsc", pasta=PASTA_PROCESSED)

print(f"\nCandidatos PV:   {len(cand_pv)}")
print(f"Candidatos IBSC: {len(cand_ibsc)}")

# ── Prévia dos top candidatos PV com estrutura confirmada ─────────────────────
if len(cand_pv) > 0:
    top = cand_pv[cand_pv["estrutura_esperada"].isin(["perovskita dupla","kesterita"])]
    print(f"\n── Top 10 candidatos PV com estrutura confirmada (n={len(top)}) ──")
    cols_preview = ["formula","familia","band_gap","is_gap_direct",
                    "energy_above_hull","crystal_system","estrutura_esperada"]
    cols_preview = [c for c in cols_preview if c in top.columns]
    print(top[cols_preview].head(10).to_string(index=False))

## 11. Resumo dos Arquivos Gerados

```
figures/
├── fig_gap_por_estrutura.png        ← novo: gap por categoria estrutural
├── fig1_distribuicao_gap.png
├── fig2_estabilidade.png
├── fig3_formacao_vs_gap.png
├── fig4_sistemas_cristalinos.png
├── fig5_efermi.png                  ← substitui work_function + efermi
├── fig6_correlacoes.png
└── fig7_candidatos_scatter.png

data/processed/
├── perovskita.csv                   ← atualizado com estrutura_esperada
├── kesterita.csv                    ← atualizado com estrutura_esperada
├── candidatos_pv.csv                ← inclui estrutura_esperada e efermi
└── candidatos_ibsc.csv
```

## 12. Próximos Passos — Fase 2

**Decisão imediata:** com base na Seção 7, definir se as análises seguintes
usam apenas os materiais com `estrutura_esperada == "perovskita dupla"` / `"kesterita"`
ou se mantêm todas as categorias com estratificação explícita.

**Fase 2 — Descritores composicionais (Matminer/Magpie):**
```python
from matminer.featurizers.composition import ElementProperty
from pymatgen.core import Composition

# Carregar apenas estruturas confirmadas
df = carregar("perovskita")
df = df[df["estrutura_esperada"] == "perovskita dupla"]

df["composition"] = df["formula"].apply(Composition)
ep = ElementProperty.from_preset("magpie")   # 114 features
df = ep.featurize_dataframe(df, "composition", ignore_errors=True)
```

**`possible_species` — reservado para Fase 2/3:**
Após restringir ao subconjunto estruturalmente confirmado, usar
`possible_species` para verificar balanço de cargas e como feature
adicional nos modelos de ML.